## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-12-00-53-40 +0000,nyt,Blue Texas? Battleground Kansas? Both Parties ...,https://www.nytimes.com/2026/11/11/us/blue-tex...
1,2026-11-12-00-53-16 +0000,nyt,Nvidia Shares Plummet as Board Is Said to Cons...,https://www.nytimes.com/2026/11/11/business/nv...
2,2026-11-12-00-53-14 +0000,nyt,Soldier Shouts Toward Hegseth to Show ‘Respect...,https://www.nytimes.com/2026/11/11/us/politics...
3,2026-11-12-00-52-57 +0000,bbc,Democrat Scott Colom seals historic win in Mis...,https://www.bbc.co.uk/news/articles/cw7vppyr80...
4,2026-11-12-00-52-41 +0000,nyt,Bruce Giudici Is on the Cusp of Victory in Thi...,https://www.nytimes.com/2026/11/11/us/californ...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2444/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
320,trump,33
9,nvidia,19
401,war,18
434,house,17
91,democrats,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
89,2026-11-11-22-20-00 +0000,wsj,Vice President JD Vance is considering face-to...,https://www.wsj.com/world/middle-east/jd-vance...,157
84,2026-11-11-22-25-23 +0000,nypost,Nvidia’s Jensen Huang also had alleged affair ...,https://nypost.com/2026/11/11/business/nvidias...,128
130,2026-11-11-21-13-00 +0000,wsj,"Democrats, fresh off sweeping midterm wins, ar...",https://www.wsj.com/politics/national-security...,121
296,2026-11-11-11-53-00 +0000,wsj,House Republicans saved Pete Hegseth and picke...,https://www.wsj.com/politics/policy/house-gop-...,120
398,2026-11-11-02-00-00 +0000,wsj,Lawmakers are demanding tighter restrictions o...,https://www.wsj.com/politics/policy/jensen-hua...,108


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
89,157,2026-11-11-22-20-00 +0000,wsj,Vice President JD Vance is considering face-to...,https://www.wsj.com/world/middle-east/jd-vance...
84,128,2026-11-11-22-25-23 +0000,nypost,Nvidia’s Jensen Huang also had alleged affair ...,https://nypost.com/2026/11/11/business/nvidias...
130,105,2026-11-11-21-13-00 +0000,wsj,"Democrats, fresh off sweeping midterm wins, ar...",https://www.wsj.com/politics/national-security...
25,76,2026-11-12-00-09-00 +0000,wsj,Democrat Scott Colom is the first Black senato...,https://www.wsj.com/politics/elections/scott-c...
296,60,2026-11-11-11-53-00 +0000,wsj,House Republicans saved Pete Hegseth and picke...,https://www.wsj.com/politics/policy/house-gop-...
72,57,2026-11-11-22-39-00 +0000,wsj,U.S. Treasury Secretary Scott Bessent was rece...,https://www.wsj.com/politics/national-security...
393,51,2026-11-11-02-09-00 +0000,wsj,The Kennedy family is joining the efforts to b...,https://www.wsj.com/politics/policy/kennedy-fa...
254,50,2026-11-11-14-40-48 +0000,nypost,Joe Biden claims he would have resigned if he ...,https://nypost.com/2026/11/11/us-news/joe-bide...
345,50,2026-11-11-09-47-59 +0000,nypost,Bernie Sanders won’t support Schumer for Senat...,https://nypost.com/2026/11/11/us-news/bernie-s...
144,42,2026-11-11-20-07-33 +0000,nypost,LA City Council could meet just one day a week...,https://nypost.com/2026/11/11/us-news/la-city-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
